# CS 1066 Classnotes (20261111)

### As You Get Settled

1.  Start your CS1066 codespace. Go to `https://github.com/<your_username>/cs1066`, and either launch a new or restart the existing codespace for this repo.

2.  In this codespace, run `python3 grab1066.py m15b`. If you get an error, run `cd /workspaces/cs1066` in the terminal and then rerun the grab command.

3.  To get ready for the quiz, set your cellphone to silent, put your electronics away, and grab a pencil/pen. We'll post the Qwickly Attendance QR code after the quiz.

----
----

### Today's topics:

1.  Quiz 8
2.  Continue building off Sid's guest lecture with a new focus on parallelism
3.  Understanding execution order
4.  Rebuilding the "roommate split" workflow

----
----

### Concurrent Execution

In talking about "tool_use" blocks, the Claude Platform Docs (CPD) stated the following:

> Claude may include several "tool_use" blocks in a single response. "How you run those calls is your decision. The API doesn’t prescribe an execution order: you can run the calls concurrently ..., sequentially in the order they appear, or in any combination that suits your tools." [\[REF\]](https://platform.claude.com/docs/en/agents-and-tools/tool-use/parallel-tool-use#execution-semantics)

Oh, how nice, Anthropic is giving us lots of freedom, right?

.

..

...

.... PLEASE DON'T SCROLL DOWN UNTIL TOLD TO DO SO

...

..

.

Ha! No. Anthropic (and every other AI provider that does this) is punting the hard problem to you. Today's goal is to understand why this problem is hard.

----

### What is "an execution order"?

Let's tackle each part of this CPD quote, and we'll start with the phrase "execution order."

To understand this term, recall what **an algorithm** is: a sequence of well-specified steps that when executed allow a deterministic program (or us acting like one) to solve a problem.

-   Let's assume our problem is to find the area of a circle, and we know that we can write a Python program to solve it:

In [ ]:
### m15/area.py

import math

ans = input("What is the radius of your circle? ")

radius = float(ans)
tmp = radius ** 2
area = math.pi * tmp

print(f"The area of a circle with radius {radius} is {area}")

-   Can you swap lines 8 and 9 in this program?

.

..

...

.... PLEASE DON'T SCROLL DOWN UNTIL TOLD TO DO SO

...

..

.

-   No, and the reason is that line 8 must execute before line 9 in order for the program to compute a correct result.
-   Why is the result incorrect if you swap lines 8 and 9? Because line 9 depends upon the result (`tmp`) produced on line 8.
-   The ordering of lines 8 and 9 is fixed because of a **data dependency**. Line 9's execution depends on the data produced on line 8.
-   In fact, every line has a data dependency on the previous line except for one. Which one?

----

### Sequential Execution

Sequential = run 1 statement/task at a time in some specified order

-   E.g., statements 5, 7, 8, 9, and 11 in `area.py` must be run sequentially in that order.

Let's consider the impact of data dependencies on the operation of loops.

In [ ]:
### m15/factorial.py

ans = input("What n would you like to compute the factorial of? ")
n = int(ans)

fact = 1
for i in range(1, n + 1):
    fact *= i

print(f"The factorial of {n} is {fact}")

-   There's one line 8 statement in `factorial.py`, but for a sufficiently large `n`, this statement executes many times.
-   Does the execution of line 8 in iteration `i+1` have to follow that line's execution in iteration `i`?

.

..

...

.... PLEASE DON'T SCROLL DOWN UNTIL TOLD TO DO SO

...

..

.

-   Yes, the value of `fact` computed in iteration `i+1` depends on the value of `fact` computed in iteration `i`.
-   `fact` is a **loop-carried dependency** in this implementation of factorial.

----

### Parallel Execution

Parallel = run 2 or more statements/tasks at a time (i.e., their execution order doesn't matter)

-   E.g., statements 4 and 6 in `factorial.py` may run in any order, or at the same time.
-   More interesting is **coarse-grained** parallelism, which concurrently runs multiple blocks of statements of a large task.
-   NOTE: `sepia.py` requires a virtual environment and `requirements.txt` installed.

In [ ]:
### m15/sepia.py

from PIL import Image

imfile = 'images/' + input('Filename of input image: ')

with Image.open(imfile) as im:
    im = im.convert('RGB') # convert to RGB if not already

    for i in range(im.size[0]):
        for j in range(im.size[1]):
            pixel = im.getpixel((i,j))

            # Give the pixel a sepia tone
            r, g, b = pixel
            tr = int(0.393 * r + 0.769 * g + 0.189 * b)
            r = 255 if tr > 255 else tr
            tg = int(0.349 * r + 0.686 * g + 0.168 * b)
            g = 255 if tg > 255 else tg
            tb = int(0.272 * r + 0.534 * g + 0.131 * b)
            b = 255 if tb > 255 else tb
            new_pixel = (r, g, b)

            im.putpixel((i,j), new_pixel)

    im.save('images/sepia.png')

-   Can the body of the inner for-loop (i.e., loop on `j`, lines 12-24) run in parallel?
-   Can the body of the outer for-loop (i.e., loop on `i`, lines 11-24) run in parallel?

.

..

...

.... PLEASE DON'T SCROLL DOWN UNTIL TOLD TO DO SO

...

..

.

-   Yes, to both questions! You can change the pixels in any order or all in parallel. You can also change each column of pixels concurrently. Parallelizing the columns is coarse-grained parallelism compared to parallelizing the changing of the image's individual pixels.

----
----

### Apply This Knowledge to the "Roommate Split" Workflow

The "roommate split" workflow (Sid's slides 25-26) involves:

-   Inputs
    - Receipt photos (any number, not just 5)
    - House rules (at least 3 types)
    
-   Outputs (simplified)
    - Balance sheet of who owes whom, which we'll implement as a CSV instead of a Markdown file.
    - Draft payment messages, which our team won't create. Why? Because, for example, Venmo has no API for sending payments, and we might as well let the humans write their own short notes on the Venmo payments.

As I've already started to do, let's continue to improve the workflow!

Questions:

1.  Is there work that can be done in parallel? If yes, what is it?
2.  What work is best done sequentially? Why?

----

### Gotchas

Things we learned about working with a single model seem to be true when working with AI to create a workflow for multiple small and cheap models.

-   What assumptions did AI make? Are they the assumptions that we would have made?
-   Will the code that it creates work in every situation?
-   Parallelism and synchronization add new ways for AI-generated code to fail.

**My recommendation:**

-   Think through the workflow before you ask AI to build it.
-   Learn more about concurrency patterns and what makes them difficult to implement correctly.

----

### Patterns and the Job's Shape

Look at Slide 7.

-   Which are showing parallelism?
-   It's not always obvious.

There's a lot more to learn about parallel processing. Dig in as you attempt to craft concurrent workflows.

----
----

### Commit Your Notes and Edits

Don't forget to review, commit, and push your edits so that you've updated your GitHub repo with today's work.

*   `git status` to review what's new and what's been edited.
*   `git add <list_of_files>` to indicate what you want committed.
*   `git commit -m "<your_commit_message>"` to commit the changes to the codespace's filesystem.
*   `git push` to push those committed changes to your GitHub repo.
*   `git status` to verify that you haven't forgotten anything.

Version 20261110